### Programming for Science and Finance

*Prof. Götz Pfeiffer, School of Mathematical and Statistical Sciences, University of Galway*

# Notebook X: Programming for Finance II

This notebook accompanies **Part III: Programming for Finance**. You will:

* apply **numerical differentiation** to the Black–Scholes formula to compute
  option **sensitivities** (the Greeks: Delta, Gamma, Vega)
* use the same finite-difference approach to compute the **duration** and **convexity**
  of a bond, and see how they approximate price changes under yield shifts
* simulate **geometric Brownian motion** as a stochastic version of the growth ODE
  from the previous notebook
* price a European option by **Monte Carlo simulation** and verify convergence
  to the Black–Scholes formula

By the end of this notebook, you will see how randomness, numerical calculus, and root-finding combine in a single workflow.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

## Task 1: The Greeks — Sensitivities of Options.

Recall the **Black–Scholes price** of a European call option:
$$
C_{\mathrm{BS}}(S, K, r, \sigma, T)
= S\,N(d_1) - K e^{-rT} N(d_2),
$$
with $d_1 = \dfrac{\ln(S/K) + (r + \frac12\sigma^2)T}{\sigma\sqrt{T}}$
and $d_2 = d_1 - \sigma\sqrt{T}$.

A trader holding an option is exposed to changes in the market: the stock price moves,
time passes, volatility shifts.  
The **Greeks** measure how sensitive the option price is to each of these variables.  
They are simply **partial derivatives** of $C_{\mathrm{BS}}$:

| Greek | Definition | Meaning |
|-------|------------|---------|
| Delta $\Delta$ | $\partial C / \partial S$ | price change per \$1 move in stock |
| Gamma $\Gamma$ | $\partial^2 C / \partial S^2$ | rate of change of Delta |
| Vega $\mathcal{V}$ | $\partial C / \partial \sigma$ | price change per unit increase in volatility |

The standard normal CDF $\Phi(x) = \frac12(1 + \mathrm{erf}(x/\sqrt{2}))$ and the Black--Scholes pricing function were developed in Notebook 9 and collected in a module `blackscholes.py`, so that they can simply be imported here (Notebook 4, Task 7).

In [ ]:
from blackscholes import phi, BS_price

print(BS_price.__doc__)

### Numerical Differentiation.

Recall (from Notebook 8) the **central difference** approximation:
$$
f'(x) \approx \frac{f(x+h) - f(x-h)}{2h},
\qquad
f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}.
$$
These formulas work for **any** differentiable function — including $C_{\mathrm{BS}}$.
We only need to choose which variable to differentiate with respect to.

### Delta.

Delta is the derivative of $C$ with respect to the stock price $S$:
$$
\Delta = \frac{\partial C}{\partial S}
\approx \frac{C_{\mathrm{BS}}(S+h,\ldots) - C_{\mathrm{BS}}(S-h,\ldots)}{2h}.
$$
Delta tells us how many dollars the option price moves for each dollar the stock moves.
For a call, $0 \le \Delta \le 1$.

In [ ]:
def delta(S, K, r, sigma, T, h=0.01):
    return (BS_price(S + h, K, r, sigma, T) - BS_price(S - h, K, r, sigma, T)) / (2 * h)

In [ ]:
# Example parameters
S, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0

print(f"Option price : {BS_price(S, K, r, sigma, T):.4f}")
print(f"Delta        : {delta(S, K, r, sigma, T):.4f}")

A Delta of $0.64$ says that if the stock rises by one euro, the option gains about $64$ cents: the option follows the stock, but not one for one.

### Gamma.

Gamma is the **second derivative** of $C$ with respect to $S$:
$$
\Gamma = \frac{\partial^2 C}{\partial S^2}
\approx \frac{C_{\mathrm{BS}}(S+h,\ldots) - 2C_{\mathrm{BS}}(S,\ldots) + C_{\mathrm{BS}}(S-h,\ldots)}{h^2}.
$$
Gamma measures the **curvature** of the option price as a function of the stock price.
A high Gamma means Delta changes rapidly — the position needs frequent rebalancing.

In [ ]:
def gamma(S, K, r, sigma, T, h=0.01):
    return (BS_price(S + h, K, r, sigma, T)
            - 2 * BS_price(S, K, r, sigma, T)
            + BS_price(S - h, K, r, sigma, T)) / h**2

In [ ]:
print(f"Gamma        : {gamma(S, K, r, sigma, T):.4f}")

### Vega.

Vega is the derivative of $C$ with respect to volatility $\sigma$:
$$
\mathcal{V} = \frac{\partial C}{\partial \sigma}
\approx \frac{C_{\mathrm{BS}}(\ldots, \sigma+h, \ldots) - C_{\mathrm{BS}}(\ldots, \sigma-h, \ldots)}{2h}.
$$
Vega is typically quoted per 1% change in volatility, so we divide by 100.

In [ ]:
def vega(S, K, r, sigma, T, h=1e-4):
    return (BS_price(S, K, r, sigma + h, T) - BS_price(S, K, r, sigma - h, T)) / (2 * h)

In [ ]:
print(f"Vega (per unit σ) : {vega(S, K, r, sigma, T):.4f}")
print(f"Vega (per 1% σ)   : {vega(S, K, r, sigma, T) / 100:.4f}")

### Plotting the Greeks as functions of stock price.

All three Greeks vary as the stock price moves.  
Delta increases from 0 (deep out-of-the-money) to 1 (deep in-the-money).  
Gamma peaks near the strike price, where uncertainty is highest.

In [ ]:
spots = np.linspace(60, 140, 200)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(spots, [delta(s, K, r, sigma, T) for s in spots])
axes[0].axvline(K, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title('Delta'); axes[0].set_xlabel('Stock price S')

axes[1].plot(spots, [gamma(s, K, r, sigma, T) for s in spots])
axes[1].axvline(K, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Gamma'); axes[1].set_xlabel('Stock price S')

axes[2].plot(spots, [vega(s, K, r, sigma, T) for s in spots])
axes[2].axvline(K, color='gray', linestyle='--', linewidth=0.8)
axes[2].set_title('Vega'); axes[2].set_xlabel('Stock price S')

plt.tight_layout()
plt.show()

---
**Exercises.**

1. The **Theta** of an option is $\Theta = \partial C / \partial T$ — the change in price as
   time passes.  Implement `theta(S, K, r, sigma, T)` numerically.  
   Why would you expect $\Theta < 0$ for a call option?

2. The **Rho** is $\partial C / \partial r$ — sensitivity to the interest rate.
   Implement `rho(S, K, r, sigma, T)` numerically.

3. The analytical formula for Delta is $\Delta = N(d_1) = \Phi(d_1)$, where $d_1$ is as in
   the Black–Scholes formula.  Verify that your numerical Delta matches `phi(d1)` for
   several values of $S$.

4. Plot all five Greeks (Delta, Gamma, Vega, Theta, Rho) against the stock price $S$
   in a single figure with five subplots.

5. Fix $S = K = 100$ and plot Delta as a function of time to maturity $T \in [0.1, 2]$.
   What happens to Delta as the option approaches expiry?

---

## Task 2: Bond Duration and Convexity.

Recall from the previous notebook: the **theoretical price** of a bond with face value $F$,
coupon $C$, $n$ periods, and yield $y$ is
$$
P(y) = \sum_{k=1}^{n} \frac{C}{(1+y)^k} + \frac{F}{(1+y)^n}.
$$

A bond portfolio manager needs to know: if the yield rises by 0.1%, how much does the
portfolio value change?  The answer lies in the **derivatives** of $P(y)$.

**Duration** and **convexity** are the bond analogues of Delta and Gamma for options.

In [ ]:
def bond_price(y, C, F, n):
    k = np.arange(1, n + 1)
    return np.sum(C / (1 + y)**k) + F / (1 + y)**n

### Modified Duration.

The **modified duration** $D$ measures the **percentage price change** per unit yield change:
$$
D = -\frac{1}{P(y)} \frac{dP}{dy}
\approx -\frac{1}{P(y)} \cdot \frac{P(y+h) - P(y-h)}{2h}.
$$
A bond with duration $D = 7$ years loses approximately $7\%$ in value
if yields rise by $1\%$.

In [ ]:
def modified_duration(y, C, F, n, h=1e-5):
    P = bond_price(y, C, F, n)
    dP = (bond_price(y + h, C, F, n) - bond_price(y - h, C, F, n)) / (2 * h)
    return -dP / P

In [ ]:
C, F, n = 30, 1000, 20
y = 0.036   # yield close to the YTM found in NB9

P0 = bond_price(y, C, F, n)
D  = modified_duration(y, C, F, n)
print(f"Bond price       : {P0:.2f}")
print(f"Modified duration: {D:.4f} years")

A modified duration of $14.55$ says that a rise of one percentage point in the yield costs roughly $14.5\%$ of the price, here about EUR $133$.  A long bond with a small coupon is that sensitive.

### Convexity.

The **convexity** $\mathcal{X}$ is the **second derivative** of $P(y)$, again normalised by $P$:
$$
\mathcal{X} = \frac{1}{P(y)} \frac{d^2 P}{dy^2}
\approx \frac{1}{P(y)} \cdot \frac{P(y+h) - 2P(y) + P(y-h)}{h^2}.
$$
Convexity corrects the duration approximation for large yield moves.

In [ ]:
def convexity(y, C, F, n, h=1e-4):
    P = bond_price(y, C, F, n)
    d2P = (bond_price(y + h, C, F, n)
           - 2 * P
           + bond_price(y - h, C, F, n)) / h**2
    return d2P / P

In [ ]:
X = convexity(y, C, F, n)
print(f"Convexity        : {X:.4f}")

### Price approximation using duration and convexity.

Given a yield shift $\Delta y$, the **Taylor expansion** gives
$$
P(y + \Delta y) \approx P(y)\,\bigl(1 - D\,\Delta y + \tfrac12 \mathcal{X}\,(\Delta y)^2\bigr).
$$
Using only the first-order (duration) term is inaccurate for large moves;
adding the second-order (convexity) term greatly improves the estimate.

In [ ]:
dy_range = np.linspace(-0.03, 0.03, 200)

P_exact    = np.array([bond_price(y + dy, C, F, n) for dy in dy_range])
P_duration = P0 * (1 - D * dy_range)
P_convex   = P0 * (1 - D * dy_range + 0.5 * X * dy_range**2)

plt.plot(dy_range * 100, P_exact,    label='Exact')
plt.plot(dy_range * 100, P_duration, '--', label='Duration only')
plt.plot(dy_range * 100, P_convex,   ':', label='Duration + convexity')
plt.xlabel('Yield shift (%)'); plt.ylabel('Bond price')
plt.legend(); plt.title('Price approximations for yield shifts')
plt.show()

* The duration-only approximation is a **straight line** tangent to the price-yield curve.
* Adding convexity captures the **curvature** and gives a much better estimate for large moves.
* This is exactly the same as using a first- vs. second-order Taylor approximation.

---
**Exercises.**

1. Compute the modified duration and convexity for several bonds with different maturities
   $n = 5, 10, 20, 30$ years (keep $C = 30$, $F = 1000$, $y = 0.04$).  
   How does duration depend on maturity?

2. The **dollar duration** is $\mathrm{DD} = -dP/dy \approx D \cdot P(y)$.  
   Compute it and verify: if the yield rises by $0.01$ (1 percentage point),
   the bond price falls by approximately $\mathrm{DD} \times 0.01$.

3. For the bond above ($C=30$, $F=1000$, $n=20$, $y=0.036$), use `BisectionSolver`
   from the `rootfinder` module (`from rootfinder import BisectionSolver`, see Notebook 9) to find the YTM when the market price is $P = 950$.  
   Then compute the duration and convexity at that yield.

4. A **zero-coupon bond** pays only the face value $F$ at maturity: $P(y) = F/(1+y)^n$.  
   Show analytically that its modified duration equals $n/(1+y)$, and verify numerically.

---

## Task 3: Random Walks and Monte Carlo Simulation.

In Notebook 9 we studied the ODE
$$
\frac{dS}{dt} = r S(t),
$$
whose solution is $S(t) = S_0 e^{rt}$ — the continuously compounded growth model.  
Real stock prices, however, do not grow smoothly: they fluctuate randomly at every moment.

The standard model for stock price randomness is **Geometric Brownian Motion** (GBM),
the **stochastic** version of the same ODE:
$$
dS = r S\,dt + \sigma S\,dW,
$$
where $dW$ is a random term with variance $dt$.  
The second term adds random noise proportional to the current price.

Over a small time step $\Delta t$, a single step of this process is:
$$
S(t + \Delta t) = S(t) \cdot \exp\!\Bigl((r - \tfrac12\sigma^2)\Delta t
                        + \sigma\sqrt{\Delta t}\; Z\Bigr),
\quad Z \sim N(0,1).
$$
Note that this step is **exact**: unlike Euler's method in Notebooks 8 and 9, the formula above is the exact solution of the stochastic equation over the interval $\Delta t$, and it introduces no discretisation error.  This is why the terminal price $S_T$ below can be simulated in a single step.

In code it is the **exponent** that is accumulated, not the price.  Taking logarithms turns the step above into a sum, $\log S(t + \Delta t) = \log S(t) + (r - \tfrac12\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\, Z$, so the log-price after $k$ steps is $\log S_0$ plus the first $k$ of these terms -- and `np.cumsum` computes all those partial sums in one sweep.  Exponentiating turns the result back into prices, and `np.hstack` puts the starting price $S_0$ back in front of each path.

The random numbers are drawn as a single array `Z` of shape `(n_paths, n_steps)`, so every path is simulated at the same time, with no loop over paths.

In [ ]:
def gbm_paths(S0, r, sigma, T, n_steps, n_paths, rng):
    """Simulate n_paths GBM paths with n_steps time steps each."""
    dt = T / n_steps
    Z = rng.standard_normal((n_paths, n_steps))
    log_returns = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_S = np.log(S0) + np.cumsum(log_returns, axis=1)
    return np.hstack([np.full((n_paths, 1), S0), np.exp(log_S)])

In [ ]:
S0, r, sigma, T = 100, 0.05, 0.2, 1.0
rng = np.random.default_rng(42)

paths = gbm_paths(S0, r, sigma, T, n_steps=252, n_paths=50, rng=rng)
times = np.linspace(0, T, paths.shape[1])

plt.plot(times, paths.T, alpha=0.4, linewidth=0.8)
plt.plot(times, S0 * np.exp(r * times), 'k--', linewidth=2, label=r'$S_0 e^{rt}$ (mean)')
plt.xlabel('Time (years)'); plt.ylabel('Stock price')
plt.title('Geometric Brownian Motion: 50 simulated paths')
plt.legend(); plt.show()

* Each path looks like a **random walk** that grows on average at rate $r$.
* The spread of paths widens over time, controlled by $\sigma$.
* The dashed curve $S_0 e^{rt}$ is the **expected** stock price — the deterministic skeleton.

### Monte Carlo Option Pricing.

A **European call option** pays $\max(S_T - K, 0)$ at time $T$.  
By the risk-neutral pricing formula, its price is the **discounted expected payoff**:
$$
C = e^{-rT}\,\mathbb{E}[\max(S_T - K, 0)].
$$
We can estimate this expectation by the **Law of Large Numbers**:
simulate $N$ independent terminal prices $S_T^{(1)}, \ldots, S_T^{(N)}$,
compute the payoff for each, and take the average.

Since we only need the terminal price $S_T$ (not the full path), the simulation simplifies to a
single step:
$$
S_T = S_0 \cdot \exp\!\Bigl((r - \tfrac12\sigma^2)T + \sigma\sqrt{T}\, Z\Bigr),
\quad Z \sim N(0,1).
$$

In [ ]:
def mc_call_price(S0, K, r, sigma, T, N, rng):
    Z  = rng.standard_normal(N)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(ST - K, 0)
    return np.exp(-r * T) * payoffs.mean()

In [ ]:
K = 100
rng = np.random.default_rng(42)

bs  = BS_price(S0, K, r, sigma, T)
mc  = mc_call_price(S0, K, r, sigma, T, N=100_000, rng=rng)

print(f"Black–Scholes price : {bs:.4f}")
print(f"Monte Carlo price   : {mc:.4f}")
print(f"Difference          : {abs(mc - bs):.4f}")

### Convergence of the Monte Carlo estimator.

As $N$ grows, the Monte Carlo estimate converges to the Black–Scholes price
by the Law of Large Numbers.  
The typical error decreases as $1/\sqrt{N}$ — doubling accuracy requires four times as many simulations.

In [ ]:
Ns   = [100, 500, 1000, 5000, 10_000, 50_000, 100_000]
rng  = np.random.default_rng(7)
mcs  = [mc_call_price(S0, K, r, sigma, T, n, rng) for n in Ns]

plt.semilogx(Ns, mcs, 'o-', label='MC estimate')
plt.axhline(bs, color='red', linestyle='--', label='Black–Scholes')
plt.xlabel('Number of simulations N'); plt.ylabel('Option price')
plt.title('Monte Carlo convergence to Black–Scholes')
plt.legend(); plt.show()

* For small $N$, the estimate fluctuates widely around the true price.
* By $N = 100{,}000$, the error is typically a few hundredths: the **standard error** of the estimate is $\mathrm{sd}(\text{payoff})/\sqrt{N} \approx 14.7/316 \approx 0.05$.
* The Black–Scholes formula is exact; Monte Carlo is an approximation — but it generalises
  to situations where no analytical formula exists.

---
**Exercises.**

1. A **European put option** pays $\max(K - S_T, 0)$ at maturity.  
   Write a function `mc_put_price` and price the put with the same parameters.  
   Compare to the Black–Scholes put formula:
   $$P_{\mathrm{BS}} = K e^{-rT} N(-d_2) - S N(-d_1).$$

2. **Put–call parity** states that $C - P = S_0 - K e^{-rT}$ for European options
   with the same strike and maturity.  Verify this numerically using your Monte Carlo
   estimates of $C$ and $P$.

3. Estimate the Monte Carlo error more carefully.  For a fixed $N = 10{,}000$,
   run the simulation $M = 100$ times (each with a different random seed)
   and compute the standard deviation of the resulting prices.  
   How does the standard deviation scale with $N$?

4. A **digital call option** pays $\$1$ if $S_T > K$ and $\$0$ otherwise.  
   Price it by Monte Carlo.  What is the expected result in terms of $\Phi(d_2)$?

5. Plot the distribution of $S_T$ for your simulation as a histogram (`plt.hist`),
   using a log scale on the $x$-axis.  What shape do you expect, and why?

---

## Summary.

This notebook extended the financial applications of Notebook 9 in three directions,
each rooted in a technique from the numerical methods notebooks.

* We computed the **Greeks** — Delta, Gamma, Vega — by applying **finite difference formulas**
  directly to the Black–Scholes function, without requiring any analytical differentiation.
* We computed **bond duration and convexity** by the same approach, and saw how the
  Taylor expansion with these two quantities approximates bond price changes under yield shifts.
* We introduced **Geometric Brownian Motion** as a stochastic version of the exponential growth
  ODE, and used it to price a European call option by **Monte Carlo simulation**.
* We verified that the Monte Carlo price converges to the Black–Scholes price,
  illustrating the **Law of Large Numbers** in a financial context.

Together, these examples show that numerical differentiation, ODE solving, and simulation
are not isolated techniques but a coherent toolkit at the heart of quantitative finance.